In [3]:
import jsonlines
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
import numpy as np
import nltk
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import regex as re
from collections import Counter

In [4]:
data_list = []

In [5]:
with jsonlines.open("data/train.jsonl", mode='r') as reader:
    for obj in reader:
        obj.pop('model')
        obj.pop('genre')
        obj.pop('id')
        data_list.append(obj)

In [6]:
len(data_list)

23707

In [7]:
df = pd.DataFrame(data_list)

In [8]:
print(df)

                                                    text  label
0      Duke Ellington, a titan of jazz, revolutionize...      1
1      I reflected on the shifting dynamics of media ...      1
2      In F. Scott Fitzgerald's "The Great Gatsby," t...      1
3      I still chuckle when I think about that time I...      1
4      Yoga, originating in ancient India, encompasse...      1
...                                                  ...    ...
23702  Antioch stands as a pivotal point in the narra...      1
23703  Margaret Cavendish’s The Blazing World stands ...      1
23704  In August of 1793, in a last-ditch effort to w...      0
23705  In interacting with Luciente and others, Conni...      0
23706  Meursault and Clamence, the protagonists from ...      1

[23707 rows x 2 columns]


In [9]:
tokens_all = [word_tokenize(sentence) for sentence in df.text]

In [10]:
# Removing all tokens consisting only of non-alphanumeric characters, that aren't inbetween words e.g. "battle-field",
# so any tokens such as: "''", "--", ".", "," etc. are removed, but "battle-field" is not removed. 
for sentence in tokens_all:
    while True:
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                sentence.remove(token) # For each token in a sentence, if the token consists only of 1 or more non-alphanumeric characters, remove it from the sentence.

        outs = [] # Creates an empty list. The list is reset to empty every iteration of the while-loop. 
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                outs.append(token) # This for-loop makes an additional check for any non-alphanumeric tokens, and appends then to the empty list 'outs'.

        if outs: # This checks whether the list 'outs' is empty or not. If it's not empty, then 'continue' is activated and the while-loop continues to iterate,
            continue # if the list is empty, then 'break' is activated, and the while-loop terminates, and the outer-most for-loop continues to the next 'sentence' in 'tokens_all'.
        else:
            break

In [11]:
word_list = [[word.lower() for word in sentence] for sentence in tokens_all]

In [12]:
len(word_list)

23707

In [13]:
words = []

In [14]:
for sentence in word_list:
    for word in sentence:
        words.append(word)

In [15]:
len(words)

14949644

In [16]:
vocab = sorted(set(words))

In [17]:
len(vocab)

122750

In [18]:
freq = Counter(words)

In [19]:
stop_words = set(stopwords.words('english'))

In [20]:
len(stop_words)

198

In [21]:
filtered_tokens = [word for word in words if word not in stop_words]

In [22]:
# This keeps the structure of a list with all datapoints in the train set, where each datapoint
# is in the form of a list. So it's a list with 23707 lists/data points with tokens.
# Each token has been converted to small characters with word.lower(). 
filtered_token_list = [[token for token in sentence if token not in stop_words] for sentence in word_list]
# Stop-words have now been removed as well. 

In [23]:
example = filtered_token_list[0]

In [24]:
string = " ".join(example)

In [25]:
print(string)

duke ellington titan jazz revolutionized genre innovative compositions showcasing remarkable ability integrate voice instrumental music among notable figures contributed artistic symphony ivie anderson whose scat singing mirrored improvisational prowess musicians like nanton ring dem bells ellington ingeniously interweaves scat singing dialogue saxophones creating dynamic call response underscores vision music fluid conversation voices instruments ellington consistently emphasized voice instrument equal importance traditional brass woodwinds orchestrating compositions keen ear vocal qualities approach vividly demonstrated mood indigo orchestration skills shine non-traditional chord arrangements transforming piece auditory tapestry mood color merely notes define ellington 's genius rather way orchestrates elements drawing unique talents band members like nanton hodges williams create symphony voice resonates authenticity purpose composition dusk exemplifies ellington 's adeptness captur

In [26]:
len(words)

14949644

In [27]:
len(filtered_tokens)

8018027

In [28]:
filtered_vocab = sorted(set(filtered_tokens))

In [29]:
tags = pos_tag(filtered_tokens, tagset='universal')

In [30]:
tag_list = [pos_tag(sentence, tagset='universal') for sentence in filtered_token_list]

In [31]:
len(tag_list)

23707

In [32]:
sentence_tags = [[token[-1] for token in sentence] for sentence in tag_list]

In [33]:
print(sentence_tags[:2])

[['ADJ', 'NOUN', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'ADP', 'ADJ', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'NOUN', 'PRON', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'ADP', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'VERB', 'ADV', 'NOUN', 'VERB', 'VERB', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'VERB', 'ADV', 'VERB', 'NOUN', 'NOUN', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'ADJ', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'ADV', 'VERB', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'NOUN', 'ADV', 'VERB', 'ADJ', 'NOUN', 'PRT', 'NOUN', 'ADV', 'NOUN', 'VERB', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'VERB', 'NOUN', 'ADP', 'ADJ', 'NOUN', 'VERB', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'VERB', 'PRT', 'ADJ', 'VERB', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'NOUN', 'ADJ', 'VERB', 'A

In [34]:
unique_tags = set([tag[-1] for tag in tags])

In [35]:
len(unique_tags)

12

In [36]:
tags_freq = [[Counter(sentence)] for sentence in sentence_tags]

In [63]:
example_tags = tags_freq[0][0]

In [64]:
print(example_tags)

Counter({'NOUN': 112, 'ADJ': 50, 'VERB': 47, 'ADV': 7, 'ADP': 6, 'PRT': 3, 'PRON': 1})


In [65]:
example_tags = dict(example_tags)

In [67]:
print(example_tags)

{'ADJ': 50, 'NOUN': 112, 'VERB': 47, 'ADP': 6, 'PRON': 1, 'ADV': 7, 'PRT': 3}


In [57]:
possible_tags = list(unique_tags)

In [58]:
print(possible_tags)

['ADJ', 'X', 'PRT', 'VERB', 'ADV', 'NOUN', 'ADP', 'NUM', '.', 'CONJ', 'PRON', 'DET']


In [68]:
missing_tags = [tag for tag in possible_tags if tag not in example_tags]

In [69]:
print(missing_tags)

['X', 'NUM', '.', 'CONJ', 'DET']


In [70]:
for tag in missing_tags:
    example_tags[tag] = 0

In [71]:
print(example_tags)

{'ADJ': 50, 'NOUN': 112, 'VERB': 47, 'ADP': 6, 'PRON': 1, 'ADV': 7, 'PRT': 3, 'X': 0, 'NUM': 0, '.': 0, 'CONJ': 0, 'DET': 0}


In [47]:
type(example_tags)

dict

In [72]:
# Sorting the dict, 'example_tags', by making a dict comprehension and using sorted() with lambda
# inserted in the 'key'-argument of the sorted() function.
tags_sorted = {k: v for k, v in sorted(example_tags.items(), key=lambda item: item[0])}

# key=lambda item: item[0] is exactly the same as writing:
# key = def lambda(item):
#           return item[0]

# lambda is a throwaway function, just like _ is a throwaway variable.

In [73]:
print(tags_sorted)

{'.': 0, 'ADJ': 50, 'ADP': 6, 'ADV': 7, 'CONJ': 0, 'DET': 0, 'NOUN': 112, 'NUM': 0, 'PRON': 1, 'PRT': 3, 'VERB': 47, 'X': 0}


In [52]:
print(tags_sorted.values())

dict_values([0, 50, 6, 7, 0, 0, 112, 0, 1, 3, 47, 0])


In [14]:
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=5, norm='l2', encoding="utf-8", ngram_range=(1, 2), stop_words='english')

features = tfidf.fit_transform(df.text).toarray()
labels = df.label

In [ ]:
features.shape # Now each of the 23707 sequences of text is represented by 195904 features, 
               # representing the tf-idf score for different unigrams and bigrams.

(23707, 195904)

In [28]:
model = LogisticRegression(random_state=42)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

In [31]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [34]:
print(metrics.classification_report(y_test, y_pred, labels=df['label']))

              precision    recall  f1-score   support

           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           0       0.97      0.96      0.97      1856
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97    